# KMA First Pass
Quick n dirty AI

In [1]:
import os
import json
import dspy
import numpy as np

cwd = os.getcwd()

In [ ]:
#vllm command

# CUDA_VISIBLE_DEVICES=0 vllm serve casperhansen/deepseek-r1-distill-qwen-32b-awq --max-model-len 32768 --gpu-memory-utilization 0.9 --quantization awq_marlin --port 8000
### if above fails try:#### CUDA_VISIBLE_DEVICES=0 vllm serve deepseek-ai/DeepSeek-R1-Distill-Qwen-14B --max-model-len 32768 --gpu-memory-utilization 0.9 --quantization awq_marlin --port 8000

In [7]:
vllm_model = dspy.OpenAI(
    api_base="http://localhost:8000/v1",
    api_key="not-needed",
    model="casperhansen/deepseek-r1-distill-qwen-32b-awq",
    max_tokens=1000,
    temperature=0.6
)

AttributeError: module 'dspy' has no attribute 'OpenAI'

## Demonstration with Plain Text

In [ ]:
plain_text = """
    This policy will... 
    """
# (generate example text with relevant/irrelevant metrics)

## Demonstration using RAG framework

In [ ]:
# once we have access to the vector DB through the API

In [ ]:
######
import dspy
import requests
from typing import List
from qdrant_client import QdrantClient

# --- 1. SETUP LLM & RETRIEVER ---
# Point to your vLLM server (OpenAI compatible)
vllm_model = dspy.OpenAI(
    api_base="http://localhost:8000/v1",
    api_key="not-needed",
    model="casperhansen/deepseek-r1-distill-qwen-32b-awq",
    max_tokens=1000,
    temperature=0.6
)

# Configure DSPy to use this LLM globally
dspy.settings.configure(lm=vllm_model)

# --- 2. DEFINE THE SIGNATURE ---
# This replaces your long system prompt string. 
# DSPy uses these docstrings to generate the prompt logic.
class PolicyRAGSignature(dspy.Signature):
    """
    Analyze and answer questions based strictly on legal and policy documents 
    related to peatlands, environmental governance, and sustainability.
    Maintain a professional tone and do not speculate.
    """
    context = dspy.InputField(desc="Relevant snippets from policy documents")
    question = dspy.InputField(desc="The user's query about a specific policy or regulation")
    answer = dspy.OutputField(desc="A concise summary followed by structured <h2> and <ul> sections")

# --- 3. DEFINE THE MODULE ---
class PolicyRAG(dspy.Module):
    def __init__(self, top_k=5):
        super().__init__()
        self.top_k = top_k
        # Chain of Thought (CoT) is perfect for your reasoning model
        self.generate_answer = dspy.ChainOfThought(PolicyRAGSignature)

    def forward(self, query: str, country=None, language=None):
        # A. Retrieve context from your existing Qdrant setup
        # (This uses your existing search logic from yasars_rag_main.py)
        search_results = self.get_qdrant_context(query, country, language)
        
        # B. Format context for the LLM
        context = "\n\n".join([f"Source: {res['filename']}\nText: {res['text']}" for res in search_results])
        
        # C. Generate the reasoning-based answer
        prediction = self.generate_answer(context=context, question=query)
        
        return dspy.Prediction(
            answer=prediction.answer,
            reasoning=prediction.rationale, # DSPy captures the "thinking" here
            sources=search_results
        )

    def get_qdrant_context(self, query, country, language):
        # Simplified bridge to your existing Qdrant 'search' function
        # Referencing your logic
        from yasars_rag_main import search, normalize_filter_value
        from qdrant_client.models import Filter, FieldCondition, MatchValue

        filters = []
        if country:
            filters.append(FieldCondition(key="country", match=MatchValue(value=normalize_filter_value(country))))
        if language:
            filters.append(FieldCondition(key="language", match=MatchValue(value=normalize_filter_value(language))))
        
        q_filter = Filter(must=filters) if filters else None
        return search(query, query_filter=q_filter, top_k=self.top_k)

# --- 4. EXECUTION ---
rag_pipeline = PolicyRAG(top_k=3)

# Example call
response = rag_pipeline(query="What are the restoration requirements for peatlands?")
print(f"REASONING: {response.reasoning}")
print(f"FINAL ANSWER: {response.answer}")

## Yasar's Old Code

In [ ]:
def query_llm(query, context_docs):
    if not context_docs:
        return "No relevant documents found.", []

    context_text = ""
    for idx, d in enumerate(context_docs, start=1):
        author = d.get('author', 'Unknown Author')
        filename = d.get('filename', 'Unknown')
        context_text += f"### Section {idx}: ({author}) - {filename}\n\n{d['text']}\n\n"
    context_text = context_text[:6000]

    payload = {
        "model": "mistral",
        "keep_alive": 0,
        "messages": [
            {
                "role": "system",
                "content": (

                    "You are a domain-specific AI assistant specialized in analyzing and answering questions based strictly on legal and policy documents related to peatlands, environmental governance, and sustainability."
                    "\n\nGuidelines for your response:\n"
                    "- Maintain a professional and formal tone suitable for academic, policy-making, and governmental audiences.\n"
                    "- Begin with a concise summary of the topic or policy area being addressed.\n"
                    "- Structure the response using semantic HTML: use <h2> for section headings and <ul><li> for listing key points.\n"
                    "- Focus especially on peatland-related regulations, land use classifications, climate mitigation strategies, restoration actions, and governance levels.\n"
                    "- Do NOT speculate or infer from general knowledge; respond strictly based on the provided context.\n"
                    "- Clearly attribute any cited content using author names or document titles when applicable.\n"
                    "- If the information is missing from the context, respond with: '<strong>The requested information is not found in the provided documents.</strong>'."
                )
            },
            {
                "role": "user",
                "content": f"Context:\n\n{context_text}\n\nQuestion:\n\n{query}"
            }
        ],
        "stream": False,
        "options": {
            "num_ctx": 4096,
            "num_threads": 8,
            "temperature": 0.2,
            "top_k": 40,
            "max_tokens": 600
        }
    }

    try:
        res = requests.post("http://localhost:11434/api/chat", json=payload)
        res.raise_for_status()
        data = res.json()
        return data.get("message", {}).get("content") or data.get("response", "No response from LLM."), context_docs
    finally:
        clear_memory()

def search(query, top_k=10, query_filter=None, min_score=0.3):
    if not all_docs:
        return []
    query_vector = model.encode(query).tolist()
    bm25_scores = bm25.get_scores(query.split()) if bm25 else [0] * len(all_docs)
    results = client.search(
        collection_name=COLLECTION_NAME,
        query_vector=query_vector,
        limit=top_k * 2,
        query_filter=query_filter
    )

    scored = []
    for res in results:
        text = res.payload.get("text", "")
        try:
            index = all_docs.index(text)
            bm25_score = bm25_scores[index]
        except ValueError:
            bm25_score = 0
        hybrid_score = bm25_score + res.score
        if hybrid_score >= min_score:
            scored.append({
                "text": text,
                "score": round(hybrid_score, 4),
                "record_id": res.payload.get("record_id"),
                "filename": res.payload.get("filename"),
                "author": res.payload.get("author"),
                "language": res.payload.get("language"),
                "country": res.payload.get("country")
            })

    grouped = {}
    for doc in scored:
        key = (doc["record_id"], doc["filename"], doc["author"])
        if key not in grouped or doc["score"] > grouped[key]["score"]:
            grouped[key] = doc

    return sorted(grouped.values(), key=lambda x: x["score"], reverse=True)[:top_k]